# Sentinel Sweep Dashboard
This notebook provides a visual interface for the Sweep Strategy logic, monitoring ranges and volatility spikes in real-time.

In [ ]:
import sys, os
import pandas as pd
import time
from datetime import datetime
from IPython.display import display, clear_output

# Add project root and signal directory to path
sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("../signal"))

import signal.sweep_strategy as sweep
import signal.sweep_config as config

print(f"Dashboard initialized at {datetime.now()}")
print(f"Monitoring: {config.SYMBOLS}")

### 1. Telegram Connectivity Test
Run this cell to verify that the timeout and retry logic is working and you can reach the API.

In [ ]:
print("Testing Telegram connection...")
test_msg = f"🛠 <b>Sweep Dashboard Link Active</b>\nVerified at: {datetime.now().strftime('%H:%M:%S')}"
success = sweep.send_telegram(test_msg)

if success:
    print("✅ Telegram message sent successfully!")
else:
    print("❌ Telegram failed. Check your config.py or SWEEP_TELEGRAM_BOT_TOKEN environment variables.")

### 2. Live Market Scanner
This loop will refresh every few minutes (defined in `sweep_config.py`). It shows the current technical state of each symbol.

In [ ]:
last_notified = {s: "NEUTRAL" for s in config.SYMBOLS}

try:
    while True:
        results = sweep.run_sweep_iteration(config.SYMBOLS)
        
        # Convert to DataFrame for visual display
        df_vis = pd.DataFrame(results)
        
        # Check for new signals to send to Telegram
        for res in results:
            sym = res['symbol']
            sig = res['signal']
            if sig != "NEUTRAL" and sig != last_notified[sym]:
                msg = f"🚨 {sig} {sym} @ {res['price']} | Range: {res['range_perc']}%"
                sweep.send_telegram(f"<b>{msg}</b>")
                last_notified[sym] = sig
        
        # Update UI
        clear_output(wait=True)
        print(f"Last Scan: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"Polling Interval: {config.POLLING_INTERVAL_SECONDS}s")
        display(df_vis.style.applymap(lambda x: 'background-color: #90ee90' if x == 'LONG' else ('background-color: #ffcccb' if x == 'SHORT' else ''), subset=['signal']))
        
        time.sleep(config.POLLING_INTERVAL_SECONDS)
        
except KeyboardInterrupt:
    print("Scanner stopped by user.")